The purpose of this file is to analyze and detect an up coming recession based on the data pulled from https://fred.stlouisfed.org/

In [112]:
import matplotlib.pyplot as plt
import pandas as pd
from fredapi import Fred
import datetime
import numpy as np
import random
import plotly.express as px

FRED_API_KEY = "612c90fb30af12767bbcaf9513bac5ed"
fred = Fred(api_key=FRED_API_KEY)

In [113]:
# """Tests the plotting of plotly express"""

# df = px.data.stocks()
# print(df)

# fig = px.line(
#     df,
#     x="date",
#     y=df.columns,
#     hover_data={"date": "|%B %d, %Y"},
#     title="custom tick labels",
#     template="plotly_dark",
#     width=1000,
#     height=400,
# )

# fig.update_xaxes(dtick="M1", tickformat="%b\n%Y", rangeslider_visible=True)
# fig.show()


import plotly.express as px

df = px.data.stocks(indexed=True)
fig = px.line(df)
fig.add_hline(y=0,
              
              annotation_text="Jan 1, 2018 baseline", 
              annotation_position="bottom right",
            #   annotation_font_size=20,
            #   annotation_font_color="blue"
             )
fig.add_vrect(x0="2018-09-24", x1="2018-12-18", 
              annotation_text="decline", annotation_position="top left",
              annotation=dict(font_size=20, font_family="Times New Roman"),
              fillcolor="green", opacity=0.25, line_width=0)
fig.show()

In [114]:
def get_recession_data() -> pd.DataFrame:
    """
    Creates the recession dates dataframe

    Returns:
    - pd.DataFrame: The recession dates dataframe.
    """
    df_recession = pd.read_csv("data/USREC.csv")
    df_recession = df_recession.rename(columns={"USREC": "recession", "DATE": "date"})
    df_recession["date"] = pd.to_datetime(df_recession["date"])
    df_recession = df_recession.set_index("date")
    return df_recession


recdf = get_recession_data()
recdf.reset_index(inplace=True)

fig = px.line(
    recdf,
    x="date",
    y="recession",
    hover_data={"date": "|%B %d, %Y"},
    title="custom tick labels",
    template="plotly_dark",
    width=1000,
    height=400,
)
fig.show()

In [115]:
def merge_recession_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Creates the recession dates dataframe and
    modifies the start and end dates to match the dataframe passed in.

    Parameters:
    - df (pd.DataFrame): The input dataframe containing the data.

    Returns:
    - pd.DataFrame: The modified recession dates dataframe.
    """

    df_recession = get_recession_data()

    # Make sure that both dataframes start and end at the same index
    start_date = max(df.index.min(), df_recession.index.min())
    end_date = min(df.index.max(), df_recession.index.max())

    # df = df.loc[start_date:end_date]
    df_recession = df_recession.loc[start_date:end_date]

    return df_recession

In [116]:
def get_data_after_date(
    data: pd.DataFrame, years: int = 1, days: int = 0
) -> pd.DataFrame:
    data_clean = data.dropna()

    # get the most recent date, convert to datetime obj and then get 1 year ago date
    most_recent_dt = data_clean.index[-1]
    one_year_ago_dt = most_recent_dt - pd.DateOffset(years=years, days=days)
    return data_clean.loc[one_year_ago_dt:]

In [117]:
def get_recession_start_end_list(recdf):
    reclist = []
    recdf.reset_index(inplace=True)

    # loop through the recession dataframe and find the start and end dates for each recession
    startdate, enddate = None, None
    for i in range(0, len(recdf)):
        if recdf.iloc[i, 1] == 1:  # recession detected
            startdate = recdf.iloc[i, 0]

            for j in range(i, len(recdf)):
                if recdf.iloc[j, 1] == 0:  # end of the recession
                    enddate = recdf.iloc[j, 0]
                    break

        if startdate and enddate:
            if (
                reclist and enddate == reclist[-1][1]
            ):  # if the enddate is the same as the last one, skip
                continue

            reclist.append((startdate, enddate))

            # after adding startdate and enddate to the list,
            # jump to the enddate and start iterating from there
            # i = j

            startdate, enddate = None, None
    return reclist


def plot_it(data_df, title, plot_recession_dates=False, hline=None):
    data_df.reset_index(inplace=True)
    
    fig = px.line(
        data_df,
        x="date",
        y=data_df.columns,
        hover_data={"date": "|%B %d, %Y"},
        title=title,
        template="plotly_dark",
        width=1000,
        height=400,
    )
    
    if isinstance(hline, int or float):
        fig.add_hline(y=hline, line_dash="dot", line_color="red")

    if plot_recession_dates:
        recdf = get_recession_data()

        start_date = data_df.date.min()
        end_date = data_df.date.max()

        # ensure that the recession dataframe only has dates that are in the data dataframe
        recdf = recdf.loc[(recdf.index >= start_date) & (recdf.index <= end_date)]

        for row in get_recession_start_end_list(recdf):
            x0 = str(row[0].date())
            x1 = str(row[1].date())

            fig.add_vrect(
                x0=x0,
                x1=x1,
                fillcolor="red",
                opacity=0.25,
                line_width=0,
                # annotation_text="recession",
                # annotation_position="top left",
            )
            
    # if hline:
    #     fig.add_hline(y=hline, line_dash="dot", line_color="green")
        
    fig.show()


# data_df = px.data.stocks()
# plot_it(data_df, title="Stocks", plot_recession_dates=True)

In [118]:
def plot_data(symbol, title, years, plot_recession_dates=True, hline=None):
    data = fred.get_series(symbol)
    result = get_data_after_date(data, years=years)

    if isinstance(result, pd.Series):
        result = pd.DataFrame(result).reset_index()
        result.columns = ["date", "data"]
        result = result.set_index("date")

    plot_it(result, title, plot_recession_dates=plot_recession_dates, hline=hline)

In [119]:
def test_recession_dates(use_random_timeframe: bool = False) -> None:
    """
    Ensures that data can be plotted alongside of the recession dates in one figure
    """

    end = datetime.datetime.now()
    date = datetime.datetime(1985, 1, 1)
    rec_df = get_recession_data()

    if use_random_timeframe:
        # pick a random date in the available data
        # get the first date in the recession dates
        res = random.choices(rec_df.index, k=2)
        rand_years = [i.year for i in res]
        start_year, end_year = min(rand_years), max(rand_years)
        date = datetime.datetime(start_year, 1, 1)
        end = datetime.datetime(end_year, 1, 1)

    date_list = [date]

    while date < end:
        date += datetime.timedelta(days=1)
        date_list.append(date)

    rand_data = np.random.uniform(low=0, high=1, size=len(date_list))
    data_df = pd.DataFrame({"date": date_list, "data": rand_data})

    # Tests if the recession dates were plotted correctly
    data_df = data_df.set_index("date")

    # combine the df with the recession dates
    plot_it(data_df, "Data & Recession Dates", plot_recession_dates=True)


test_recession_dates(use_random_timeframe=True)

In [120]:
# """Federal Funds Effective Rate (FEDFUNDS)"""
plot_data("FEDFUNDS", "Federal Funds Effective Rate", years=70)

In [121]:
"""
GDP Contraction: One of the primary indicators is a decline in Gross Domestic Product (GDP),
which measures the total value of goods and services produced in a country.
A negative GDP growth for two consecutive quarters is often considered a technical recession.
"""

# Real Gross Domestic Product (GDPC1)
plot_data("GDPC1", "Real GDP", years=2)

In [122]:
"""
Rising Unemployment: Job losses and rising unemployment rates are common
during a recession as businesses may cut costs by reducing their workforce.
"""

plot_data("UNRATE", "Unemployment Rate", years=2)

In [123]:
"""
Consumer spending tends to decrease during economic
downturns as people become more cautious about their finances.
This can impact various industries, especially retail.
"""

plot_data("PCE", "Consumer Spending", years=4)

In [124]:
"""
Reduced Industrial Production: A decrease in the production of goods and services
by industries is a clear sign of economic contraction.
This can be measured by the Industrial Production Index.
"""

plot_data("INDPRO", "Industrial Production", years=5)

In [125]:
"""
Stock Market Decline: Stock markets are sensitive to economic conditions.
A prolonged period of declining stock prices may indicate investor pessimism about the economic outlook.
"""

plot_data("SP500", "S&P 500", years=2)


In [126]:
"""
Housing Market Slowdown: A recession often leads to a slowdown in the real estate market,
with declining home sales, falling prices, and increased foreclosures.
"""

plot_data("CSUSHPINSA", "Housing Market", years=3)

In [127]:
"""
Tightened Credit Conditions: Banks may become more cautious about lending during economic downturns,
leading to tightened credit conditions and reduced access to loans for businesses and consumers.
"""

plot_data("DRTSCILM", "Tightened Credit Conditions", years=50)

In [128]:
"""
Inverted Yield Curve: An inverted yield curve, where short-term interest rates are higher
than long-term rates has historically been a reliable predictor of economic recessions.
"""

plot_data("T10Y2Y", "Yield Curve 10Y - 2Y", years=3, hline=0)

In [129]:
"""10-Year Treasury Constant Maturity Minus 3-Month Treasury Constant Maturity (T10Y3M)"""

plot_data("T10Y3M", "Yield Curve 10Y - 3M", years=2, hline=0)

In [130]:
"""
Business Investment Decline: Companies may cut back on investments during a recession,
leading to a decline in capital expenditures and business expansion.
"""

plot_data("W790RC1Q027SBEA", "Business Investment", years=1)

In [131]:
plot_data("CEU4348400001", "Truck Transportation Employees", years=4)

In [132]:
"""Delinquency Rate on Credit Card Loans, All Commercial Banks"""

plot_data("DRCCLACBS", "Credit Card Delinquency Rate", years=4)

In [133]:
plot_data("JTSJOL", "Job Openings", years=6)

In [134]:
"""Large Bank Consumer Credit Card Balances: Total Balances (RCCCBBALTOT)"""

plot_data(
    "RCCCBBALTOT", "Large Bank Consumer Credit Card Balances: Total Balances", years=1
)

In [135]:
"""Commercial Bank Interest Rate on Credit Card Plans, All Accounts (TERMCBCCALLNS)"""

plot_data(
    "TERMCBCCALLNS", "Commercial Bank Interest Rate on Credit Card Plans", years=2
)

In [136]:
#  Personal interest payments (B069RC1)
plot_data("B069RC1", "Personal interest payments", years=5)

Other Indicators to Consider
- Net interest and miscellaneous payments on assets
- Personal Savings Rate

In [137]:
#  Net interest and miscellaneous payments on assets (W255RC1Q027SBEA)
plot_data(
    "W255RC1Q027SBEA", "Net interest and miscellaneous payments on assets", years=10
)

In [138]:
#  Personal Saving Rate (PSAVERT)
plot_data("PSAVERT", "Personal Saving Rate", years=6)

In [139]:
# LOOK AT THESE STOCKS
# In addition, certain individual stocks have outperformed during each of the past two U.S. recessions.
# Walmart Inc. (ticker: WMT), Abbott Laboratories (ABT) and Home Depot Inc. (HD)
# are just three examples of stocks that beat the S&P 500 in both 2008 and 2020.